# 02 — Análise de Tempo Médio de Tramitação

**Objetivo:** Calcular e visualizar o tempo médio de tramitação de processos
segmentado por tribunal, classe e órgão julgador.

**Variáveis de interesse:**
- Duração em dias (data_ajuizamento → última atualização)
- Distribuição por percentis (P25, P50, P75, P90)
- Evolução temporal mensal

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datajud.collectors import coletar_dataframe
from datajud.analysis.tempo import analise_tempo_tramitacao, evolucao_mensal

sns.set_theme(style='whitegrid')

## 1. Coleta de dados

> Ajuste `tribunal`, `classe_codigo` e `max_docs` conforme necessidade.

In [ ]:
# Coletar processos do TJSP — Procedimento Comum (436)
df = coletar_dataframe(
    tribunal='tjsp',
    classe_codigo=436,
    data_inicio='2020-01-01',
    data_fim='2023-12-31',
    max_docs=500,
)

print(f'Shape: {df.shape}')
df.head()

## 2. Visão geral

In [ ]:
df.info()
df.describe()

## 3. Tempo médio de tramitação por classe

In [ ]:
stats_classe = analise_tempo_tramitacao(
    df,
    agrupar_por='classe_nome',
    top_n=15,
    plot=True,
)
stats_classe

## 4. Tempo médio por órgão julgador (top 20)

In [ ]:
stats_orgao = analise_tempo_tramitacao(
    df,
    agrupar_por='orgao_julgador',
    top_n=20,
    plot=True,
)
stats_orgao.head(10)

## 5. Evolução mensal de ajuizamentos

In [ ]:
df_mensal = evolucao_mensal(df)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    df_mensal['ano_mes'].astype(str),
    df_mensal['total_processos'],
    marker='o', linewidth=2
)
ax.set_title('Evolução mensal de ajuizamentos — TJSP')
ax.set_xlabel('Mês/Ano')
ax.set_ylabel('Processos ajuizados')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Salvar dados

In [ ]:
df.to_parquet('../data/processed/processos_tjsp.parquet', index=False)
print('Dados salvos em data/processed/processos_tjsp.parquet')